# Verificação da GPU
Confirma que a GPU NVIDIA está instalada e sendo usada pelo PyTorch (CUDA) no ambiente do projeto.

In [1]:
import subprocess, time, torch
print(subprocess.run(["nvidia-smi", "--query-gpu=name,driver_version,memory.total,memory.used,utilization.gpu,temperature.gpu",
                      "--format=csv"], capture_output=True, text=True).stdout)
p = torch.cuda.get_device_properties(0)
print(f"PyTorch {torch.__version__} | CUDA disponível: {torch.cuda.is_available()} | CUDA {torch.version.cuda} | cuDNN {torch.backends.cudnn.version()}")
print(f"GPU: {p.name} | VRAM {p.total_memory / 1024**3:.1f} GB | compute capability {p.major}.{p.minor} | bfloat16: {torch.cuda.is_bf16_supported()}")

name, driver_version, memory.total [MiB], memory.used [MiB], utilization.gpu [%], temperature.gpu
NVIDIA GeForce RTX 3050, 617.14, 8192 MiB, 970 MiB, 34 %, 49

PyTorch 2.14.1+cu130 | CUDA disponível: True | CUDA 13.0 | cuDNN 92400
GPU: NVIDIA GeForce RTX 3050 | VRAM 8.0 GB | compute capability 8.6 | bfloat16: True


In [2]:
# teste de carga: multiplicação de matrizes na GPU x CPU
a = torch.randn(4096, 4096, device="cuda", dtype=torch.bfloat16)
torch.cuda.synchronize(); t0 = time.time()
for _ in range(50):
    c = a @ a
torch.cuda.synchronize(); t_gpu = (time.time() - t0) / 50
tflops = 2 * 4096**3 / t_gpu / 1e12
b = a.float().cpu(); t0 = time.time(); b @ b; t_cpu = time.time() - t0
print(f"GPU: {t_gpu * 1000:.1f} ms por matmul 4096x4096 ({tflops:.1f} TFLOPS bf16) | CPU: {t_cpu * 1000:.0f} ms -> {t_cpu / t_gpu:.0f}x mais rápido na GPU")
print(f"Tensor em: {c.device} | memória alocada: {torch.cuda.memory_allocated() / 1024**2:.0f} MB")

GPU: 10.1 ms por matmul 4096x4096 (13.7 TFLOPS bf16) | CPU: 183 ms -> 18x mais rápido na GPU
Tensor em: cuda:0 | memória alocada: 72 MB
